# Edge-Ready Seatbelt Detection — v3.0

Real-time binary seatbelt detection (ON / OFF) optimised for edge hardware (Raspberry Pi 4).  
Two complete inference pipelines are implemented and compared side-by-side.

---

## Pipeline overview

| | Pipeline A | Pipeline B |
|---|---|---|
| **ROI extraction** | YOLOv5 full-frame detection | MediaPipe Pose (torso keypoints) |
| **Classifier** | MobileNetV3 Patch CNN | MobileNetV3 Patch CNN |
| **Smoother** | Majority-vote (5 frames) | Exponential Moving Average |
| **Best for** | Direct frontal camera | High-angle or occluded cameras |

---

## Execution order

```
1. Environment Setup   -> once
2. Data & Augmentation -> inspect transforms
3. Model Training      -> 3A (YOLO) -> 3B (CNN) -> 3C (EMA/BiLSTM)
4. Shared Utilities    -> run entire section before any pipeline
5. Pipeline A          -> YOLOv5 ROI + CNN inference loop
6. Pipeline B          -> MediaPipe + EMA inference loop
7. Pipeline Comparison -> side-by-side analysis table
8. Evaluation          -> accuracy and latency metrics
9. Export              -> ONNX / INT8 for deployment
```


## 1. Environment Setup

Install all required packages. Run once; comment out on subsequent runs.

In [1]:
!pip install -q ultralytics mediapipe albumentations onnx onnxruntime \
             opencv-python-headless torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.3 MB/s eta 0:00:00


In [ ]:
!unzip seatbelt_results3.zip

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  inflating: seatbelt_edge_v2/patches/train/OFF/4-FemaleNoGlasses_mp4-27_jpg.rf.f0e7cb5f2efb1e7486793da38029d6a5_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/IMG_20220901_182510_947_jpg.rf.7081d2e605878115b460dd3de73e0a5a_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/IMG_8079_jpeg.rf.863aea848194c1776e63ed9bfb6753d9_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/PXL_20220916_125000710_jpg.rf.9d1d83cda6b9c608f7df3ce046733076_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/IMG_8191_jpeg.rf.dadaa1b1926ebb6e872abd45b267506d_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/PXL_20220916_124948928_jpg.rf.2c1d011331d181e8983a3052d4935aad_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/IMG_8179_jpeg.rf.b5ca1c41e6ed65f3ccc3a831e972d26d_patch.jpg  
  inflating: seatbelt_edge_v2/patches/train/OFF/PXL_20220916_125044588-MP_jpg.rf.b6db6b7b83d1df1b

In [4]:
!pip install openmim


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 20.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 5.1 MB/s eta 0:00:0

In [1]:
!mim install mmdet
!mim install mmengine
!mim install mmcv

Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.12/dist-packages/mim/__init__.py", line 10, in <module>
    import setuptools  # noqa: F401
    ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py", line 16, in <module>
    import setuptools.version
  File "/usr/local/lib/python3.12/dist-packages/setuptools/version.py", line 1, in <module>
    import pkg_resources
  File "/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py", line 2172, in <module>
    register_finder(pkgutil.ImpImporter, find_on_path)
                    ^^^^^^^^^^^^^^^^^^^
AttributeError: module 'pkgutil' has no attribute 'ImpImporter'. Did you mean: 'zipimporter'?
Traceback (most recent call last):
  File "/usr/local/bin/mim", line 5, in <module>
    from mim.cli import cli
  File "/usr/local/lib/python3.12/dist-packages/mim/__init__.py", line 10, in <module>
    import setu

In [2]:
import os, sys, copy, random, warnings, time, threading, uuid
import queue as _queue
from collections import deque
from pathlib import Path
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as M
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}  |  PyTorch {torch.__version__}")


Device: cpu  |  PyTorch 2.10.0+cpu


In [3]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive; drive.mount('/content/drive')

DATASET_YAML = "/content/drive/MyDrive/seatbelt-detection.v4i.yolov8/data.yaml"
RESULTS_PATH = "/content/drive/MyDrive/seatbelt_results/seatbelt_edge_v2"
DATA_ROOT    = str(Path(DATASET_YAML).parent)
RUN_DIR      = "runs/seatbelt_edge_v2"
os.makedirs(RUN_DIR, exist_ok=True)

# ── Weights paths ─────────────────────────────────────────────────────────────
YOLO_WEIGHTS   = f"{RUN_DIR}/best.pt"
CNN_WEIGHTS    = f"{RUN_DIR}/patch_cnn.pt"
BILSTM_WEIGHTS = f"{RUN_DIR}/bilstm.pt"
CNN_ONNX_PATH  = f"{RUN_DIR}/patch_cnn.onnx"
YOLOV5_WEIGHTS = "seatbelt_edge_v2/best_github.pt"  # Pipeline A detector

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_NAMES = ['OFF', 'ON']   # 0 = no seatbelt, 1 = seatbelt
NUM_CLASSES = 2

# ── YOLOv8n training ──────────────────────────────────────────────────────────
YOLO_MODEL    = 'yolov8n.pt'
YOLO_EPOCHS   = 50
YOLO_IMGSZ    = 320           # 320px → ~15 FPS gain on RPi4 vs 640
YOLO_BATCH    = 16
YOLO_PATIENCE = 15
YOLO_AMP      = True

# ── Patch CNN training ────────────────────────────────────────────────────────
CNN_EPOCHS   = 20
CNN_LR       = 3e-4
CNN_BATCH    = 32
CNN_IMG_SIZE = 96             # 96px → 44% less compute than 128

# ── BiLSTM (optional) ─────────────────────────────────────────────────────────
SEQUENCE_LEN = 16
FUSED_DIM    = 6
LSTM_HIDDEN  = 64
LSTM_LAYERS  = 2
LSTM_EPOCHS  = 30

# ── Inference ─────────────────────────────────────────────────────────────────
CONF_THRESHOLD   = 0.45
TORSO_PAD        = 0.20       # fractional padding around MediaPipe torso ROI
USE_GEO_PRIOR    = True
MOTION_THRESH    = 8.0
SKIP_FRAMES      = 2          # carry last prediction on skipped frames

# ── Annotation colours (BGR) ──────────────────────────────────────────────────
COLORS = {'ON': (0, 200, 0), 'OFF': (0, 0, 220), 'WARMING': (160, 160, 160)}

print("Configuration loaded ✓")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configuration loaded ✓


## 2. Data & Augmentation

### Why these augmentations?

| Augmentation | Purpose |
|---|---|
| `CoarseDropout` | Simulates a steering wheel or arm partially blocking the belt |
| `RandomBrightnessContrast` | Handles night driving and harsh sunlight |
| `ShiftScaleRotate` | Accounts for different camera angles and seat positions |
| `MotionBlur` | Simulates camera shake or fast head movement |
| `HueSaturationValue` | Generalises across different seatbelt colours and fabrics |

The validation transform is resize + normalise only — no random operations.


In [4]:
def get_train_transforms(img_size: int = 96) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.6),
        A.HorizontalFlip(p=0.4),
        A.CoarseDropout(max_holes=6, max_height=20, max_width=20, min_holes=1, fill_value=0, p=0.5),
        A.OneOf([A.MotionBlur(blur_limit=(3, 9)), A.GaussianBlur(blur_limit=(3, 7))], p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.GaussNoise(var_limit=(5, 30), p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size: int = 96) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class TorsoPatchDataset(Dataset):
    '''Loads torso patches from ON/ and OFF/ subfolders.
    Run generate_torso_patches() first (Section 3B).'''
    def __init__(self, root: str, transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(CLASS_NAMES):
            folder = Path(root) / cls
            if not folder.exists():
                print(f"[WARN] Missing folder: {folder}"); continue
            for p in list(folder.glob('*.jpg')) + list(folder.glob('*.png')):
                self.samples.append((str(p), idx))
        random.shuffle(self.samples)
        print(f"TorsoPatchDataset: {len(self.samples)} samples from {root}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        return (self.transform(image=img)['image'] if self.transform else img), label

print("Augmentation pipeline ready ✓")


Augmentation pipeline ready ✓


## 3. Model Training

Three models are trained in order:

| Model | Role | Training time (Colab T4) |
|---|---|---|
| **YOLOv8n** | Detects seatbelt / classifies torso ROI (Pipeline B) | ~30 min |
| **MobileNetV3-Small** | Classifies ON/OFF on cropped patches | ~10 min |
| **EMASmoother** | No training — configured by two constants | instant |
| **BiLSTM** *(optional)* | Temporal smoother — only if labelled sequences available | ~5 min |

> Run 3A then 3B in order. 3C defines the EMA smoother (always run) and the BiLSTM class (optional).


### 3A. YOLOv8n — Seatbelt Detector

The nano variant (~3.2 M parameters) is fine-tuned at 320 × 320 px with FP16 mixed precision.  
`freeze=5` keeps the first 5 backbone layers frozen to speed up transfer learning from COCO.


In [ ]:
# ── Write custom augmentation hyp file ───────────────────────────────────────
aug_yaml = f"{RUN_DIR}/augment_hyp.yaml"
Path(aug_yaml).write_text(
    "degrees: 10.0\ntranslate: 0.1\nscale: 0.5\nfliplr: 0.5\n"
    "mosaic: 1.0\nmixup: 0.1\nerasing: 0.4\nperspective: 0.0002\n"
)

# ── Train YOLOv8n ─────────────────────────────────────────────────────────────
yolo_model = YOLO(YOLO_MODEL)
results = yolo_model.train(
    data=DATASET_YAML, epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH, patience=YOLO_PATIENCE, amp=YOLO_AMP,
    freeze=5, optimizer='AdamW', lr0=0.001, lrf=0.01,
    weight_decay=0.0005, warmup_epochs=3,
    project=RUN_DIR, name='yolo', exist_ok=True,
    plots=True, save=True, verbose=True,
)
print(f"Best weights saved → {YOLO_WEIGHTS}")


In [8]:
!git clone https://github.com/THU-MIG/yolov10.git
%cd yolov10

Cloning into 'yolov10'...
remote: Enumerating objects: 20338, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 20338 (delta 2), reused 0 (delta 0), pack-reused 20334 (from 2)
Receiving objects: 100% (20338/20338), 11.10 MiB | 15.18 MiB/s, done.
Resolving deltas: 100% (14354/14354), done.
/content/yolov10


In [19]:
!pip install git+https://github.com/THU-MIG/yolov10.git

  Cloning https://github.com/THU-MIG/yolov10.git to /tmp/pip-req-build-pjkcuw2k
  Running command git clone --filter=blob:none --quiet https://github.com/THU-MIG/yolov10.git /tmp/pip-req-build-pjkcuw2k
  Resolved https://github.com/THU-MIG/yolov10.git to commit 453c6e38a51e9d1d5a2aa5fb7f1014a711913397
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.1.34-py3-none-any.whl size=732494 sha256=d5d28262ff147d83daf80c82aa1ad7950305f98726f8c8118c427433189d6e9b
  Stored in directory: /tmp/pip-ephem-wheel-cache-yusccv8y/wheels/35/9a/d9/ec58d6096ceff21c4a77b18cd266af11385b347ca19a307597
Successfully built ultralytics
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.33
    Uninstalling ultralytics-8.4.33:
      Successfully uninstalled ultralytics-8.4.33


In [10]:
!cd yolov10

### YOLOv10n — Seatbelt Detector

The nano variant (~3.5 M parameters) is fine-tuned at 320 × 320 px with FP16 mixed precision.  
YOLOv10n uses a highly optimized backbone and head design that reduces latency while maintaining accuracy, making it ideal for **edge deployment** (e.g., in-vehicle seatbelt monitoring).  
Like YOLOv8n, early layers can be frozen during transfer learning to speed up training and leverage pretrained features from large datasets such as COCO.

In [ ]:
from ultralytics import YOLOv10

# Load pretrained model
model = YOLOv10.from_pretrained("jameslahm/yolov10n")

# Train
model.train(
    data="/content/drive/MyDrive/seatbelt-detection.v4i.yolov8/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    project="yolov10n",
    name="yolov10n",
)

New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.34 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov10n.yaml, data=/content/drive/MyDrive/seatbelt-detection.v4i.yolov8/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, val_period=1, cache=False, device=None, workers=8, project=yolov10n, name=yolov10n2, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classe

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Freezing layer 'model.23.dfl.conv.weight'


train: Scanning /content/drive/MyDrive/seatbelt-detection.v4i.yolov8/train/labels.cache... 7323 images, 0 backgrounds, 0 corrupt: 100%|██████████| 7323/7323 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /content/drive/MyDrive/seatbelt-detection.v4i.yolov8/valid/labels.cache... 696 images, 0 backgrounds, 0 corrupt: 100%|██████████| 696/696 [00:00<?, ?it/s]


Plotting labels to yolov10n/yolov10n2/labels.jpg... 
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 95 weight(decay=0.0), 108 weight(decay=0.0005), 107 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to yolov10n/yolov10n2
Starting training for 100 epochs...

      Epoch    GPU_mem     box_om     cls_om     dfl_om     box_oo     cls_oo     dfl_oo  Instances       Size


      1/100         0G      3.184      3.907      3.843      2.914      5.682      3.776         35        640:   5%|▌         | 23/458 [11:40<3:33:10, 29.40s/it]

In [ ]:
from mmengine.config import Config
from mmdet.apis import init_detector, train_detector
from mmdet.registry import DATASETS
from mmengine.runner import Runner

# ── Load base RTMDet config ────────────────────────────────────────────────
cfg = Config.fromfile(
    "configs/rtmdet/rtmdet_nano_8xb32-300e_coco.py"
)

# ── Modify dataset ─────────────────────────────────────────────────────────
cfg.dataset_type = 'CocoDataset'
cfg.data_root = 'data/seatbelt/'

cfg.train_dataloader.dataset.ann_file = 'train/annotations.json'
cfg.train_dataloader.dataset.data_prefix = dict(img='train/images/')

cfg.val_dataloader.dataset.ann_file = 'valid/annotations.json'
cfg.val_dataloader.dataset.data_prefix = dict(img='valid/images/')

cfg.test_dataloader = cfg.val_dataloader

# ── Classes ────────────────────────────────────────────────────────────────
cfg.metainfo = {
    'classes': ('seatbelt',),
    'palette': [(220, 20, 60)]
}

# ── Model adjustments ──────────────────────────────────────────────────────
cfg.model.bbox_head.num_classes = 1

# ── Training hyperparameters ───────────────────────────────────────────────
cfg.train_cfg.max_epochs = 100
cfg.train_dataloader.batch_size = 16

cfg.optim_wrapper.optimizer.type = 'AdamW'
cfg.optim_wrapper.optimizer.lr = 0.001
cfg.optim_wrapper.optimizer.weight_decay = 0.0005

# ── Augmentations (similar to your YOLO setup) ─────────────────────────────
cfg.train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='RandomResize', scale=(640, 640), ratio_range=(0.5, 2.0)),
    dict(type='RandomCrop', crop_size=(640, 640)),
    dict(type='PackDetInputs')
]

# ── Output directory ───────────────────────────────────────────────────────
cfg.work_dir = './runs/rtmdet'

# ── Train ─────────────────────────────────────────────────────────────────
runner = Runner.from_cfg(cfg)
runner.train()

### 3B. MobileNetV3-Small — Patch CNN Classifier

**Step 1 (this cell):** MediaPipe extracts torso crops from every labelled image and saves  
them into `patches/train/ON/`, `patches/train/OFF/`, `patches/valid/ON/`, `patches/valid/OFF/`.

**Step 2 (next cell):** MobileNetV3-Small (ImageNet pretrained) is fine-tuned on those crops.

> Both calls to `generate_torso_patches()` are uncommented — run this cell once only.


In [ ]:
pip uninstall mediapipe -y
pip install mediapipe==0.10.14

SyntaxError: invalid syntax (2617308986.py, line 1)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — Pose-guided torso ROI extraction
# ══════════════════════════════════════════════════════════════════════════════
import cv2
import os
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python.vision import pose_landmarker, RunningMode
from mediapipe import Image
from mediapipe import ImageFormat
# ── FIX 3: Use the lite model — ~10 ms vs ~30 ms on RPi4 ────────────────────
POSE_MODEL = "pose_landmarker_lite.task"
POSE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/"
    "pose_landmarker_lite.task"
)
if not os.path.exists(POSE_MODEL):
    print(f"Downloading {POSE_MODEL}...")
    urllib.request.urlretrieve(POSE_MODEL_URL, POSE_MODEL)
    print(f"{POSE_MODEL} downloaded.")

# -------------------------------
# 2. Initialize MediaPipe PoseLandmarker
# -------------------------------
options = pose_landmarker.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=POSE_MODEL),
    running_mode=RunningMode.IMAGE,
    min_pose_presence_confidence=0.5,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
pose_estimator = pose_landmarker.PoseLandmarker.create_from_options(options)

# -------------------------------
# 3. Torso landmark indices (MediaPipe Tasks API)
# -------------------------------
TORSO_LANDMARKS = {
    'left_shoulder' : 11,
    'right_shoulder': 12,
    'left_hip'      : 23,
    'right_hip'     : 24,
}

TORSO_PAD = 0.2  # fractional padding


def extract_torso_roi(
    frame: np.ndarray,
    padding: float = TORSO_PAD,
) -> tuple[np.ndarray | None, tuple | None, list]:
    """
    Detect body pose and return a cropped torso region (ROI).

    Parameters
    ----------
    frame   : BGR image (H×W×3)
    padding : fractional margin around the four torso landmarks

    Returns
    -------
    roi       : cropped BGR image of the torso (or None if no pose found)
    bbox      : (x1, y1, x2, y2) in pixel coordinates (or None)
    keypoints : list of (x, y) pixel coords for [L-shoulder, R-shoulder,
                                                  L-hip, R-hip]
    """
    H, W = frame.shape[:2]

    # MediaPipe requires RGB input
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = Image(image_format=ImageFormat.SRGB, data=rgb)

    # ✅ Use timestamp (must increase in video, but 0 works for offline)
    results = pose_estimator.detect(mp_image)

    if not results or not results.pose_landmarks:
        # No person detected — return full frame as fallback
        return frame, (0, 0, W, H), []

    lm_list = results.pose_landmarks[0]  # list of landmarks

    # Extract normalised (0–1) coordinates for the four torso landmarks
    pts = [
        (lm_list[TORSO_LANDMARKS['left_shoulder']].x,  lm_list[TORSO_LANDMARKS['left_shoulder']].y),
        (lm_list[TORSO_LANDMARKS['right_shoulder']].x, lm_list[TORSO_LANDMARKS['right_shoulder']].y),
        (lm_list[TORSO_LANDMARKS['left_hip']].x,       lm_list[TORSO_LANDMARKS['left_hip']].y),
        (lm_list[TORSO_LANDMARKS['right_hip']].x,      lm_list[TORSO_LANDMARKS['right_hip']].y),
    ]

    # Convert to pixel coordinates
    kp_pixels = [(int(x * W), int(y * H)) for x, y in pts]

    # Bounding box of the four keypoints
    xs = [p[0] for p in kp_pixels]
    ys = [p[1] for p in kp_pixels]
    x1_raw, y1_raw = min(xs), min(ys)
    x2_raw, y2_raw = max(xs), max(ys)

    # Add padding (shoulder-to-hip diagonal ± margin)
    pad_x = int((x2_raw - x1_raw) * padding)
    pad_y = int((y2_raw - y1_raw) * padding)

    x1 = max(0, x1_raw - pad_x)
    y1 = max(0, y1_raw - pad_y)
    x2 = min(W, x2_raw + pad_x)
    y2 = min(H, y2_raw + pad_y)

    # Crop
    roi = frame[y1:y2, x1:x2]

    # Guard against degenerate crop
    if roi.size == 0:
        return frame, (0, 0, W, H), kp_pixels

    return roi, (x1, y1, x2, y2), kp_pixels


# ── Utility: crop and save patches from a labelled image directory ────────────
def generate_torso_patches(img_dir: str, label_dir: str, out_dir: str):
    """
    Pre-compute torso patches from a YOLO-format labelled dataset.
    Saves ON/OFF subfolders of cropped torso images for the patch CNN.

    img_dir   : path to images (*.jpg / *.png)
    label_dir : path to YOLO .txt annotation files
    out_dir   : destination root (will create ON/ and OFF/ subfolders)
    """
    for cls in CLASS_NAMES:
        os.makedirs(f"{out_dir}/{cls}", exist_ok=True)

    img_paths = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))
    saved = 0

    for img_path in img_paths:
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue

        # Get corresponding label file
        label_path = Path(label_dir) / (img_path.stem + '.txt')
        if not label_path.exists():
            continue

        # Determine GT class from YOLO label (first annotation line)
        with open(label_path) as lf:
            first_line = lf.readline().strip()
        if not first_line:
            continue
        cls_idx = int(first_line.split()[0])   # 0=OFF, 1=ON
        cls_name = CLASS_NAMES[cls_idx]

        # Extract torso ROI using pose
        roi, _, _ = extract_torso_roi(frame)
        if roi is not None and roi.size > 0:
            out_path = f"{out_dir}/{cls_name}/{img_path.stem}_patch.jpg"
            cv2.imwrite(out_path, roi)
            saved += 1

    print(f"Saved {saved} torso patches → {out_dir}")


# ── FIX 1: Generate patches for BOTH train and valid ─────────────────────────
#generate_torso_patches(
#    img_dir   = f"{DATA_ROOT}/train/images",
#    label_dir = f"{DATA_ROOT}/train/labels",
#    out_dir   = f"{RUN_DIR}/patches/train",
#)
#generate_torso_patches(
#    img_dir   = f"{DATA_ROOT}/valid/images",
#    label_dir = f"{DATA_ROOT}/valid/labels",
#    out_dir   = f"{RUN_DIR}/patches/valid",
#)

print("Pose ROI extractor ready ✓")

pose_landmarker_lite.task downloaded.
Pose ROI extractor ready ✓


In [ ]:
import torchvision.models as M

def build_patch_cnn(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    '''MobileNetV3-Small with fine-tuned classification head.'''
    weights = M.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
    m = M.mobilenet_v3_small(weights=weights)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    return m

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

patch_cnn = build_patch_cnn().to(DEVICE)
print(f"MobileNetV3-Small — {count_parameters(patch_cnn):,} trainable parameters")


MobileNetV3-Small — 1,519,906 trainable parameters


In [ ]:
PATCH_TRAIN_DIR = f"{RUN_DIR}/patches/train"
PATCH_VAL_DIR   = f"{RUN_DIR}/patches/valid"

train_ds = TorsoPatchDataset(PATCH_TRAIN_DIR, transform=get_train_transforms(CNN_IMG_SIZE))
val_ds   = TorsoPatchDataset(PATCH_VAL_DIR,   transform=get_val_transforms(CNN_IMG_SIZE))
train_dl = DataLoader(train_ds, batch_size=CNN_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CNN_BATCH, shuffle=False, num_workers=2, pin_memory=True)

# ── Class-weighted loss handles dataset imbalance ─────────────────────────────
off_n = len(list(Path(PATCH_TRAIN_DIR + '/OFF').glob('*')))
on_n  = len(list(Path(PATCH_TRAIN_DIR + '/ON').glob('*')))
total = off_n + on_n
if total > 0 and off_n > 0 and on_n > 0:
    w = torch.tensor([total / (2 * off_n), total / (2 * on_n)], dtype=torch.float32).to(DEVICE)
    print(f"Class weights — OFF: {w[0]:.3f}  ON: {w[1]:.3f}")
else:
    w = None; print("[WARN] Could not compute class weights — using uniform loss")

criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
optimizer = torch.optim.AdamW(patch_cnn.parameters(), lr=CNN_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CNN_EPOCHS)
scaler    = GradScaler()
best_val_acc = 0.0
history = {'loss': [], 'val_acc': []}

for epoch in range(1, CNN_EPOCHS + 1):
    patch_cnn.train(); running_loss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(patch_cnn(imgs), labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running_loss += loss.item()

    patch_cnn.eval(); correct = total_n = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct  += (patch_cnn(imgs).argmax(1) == labels).sum().item()
            total_n  += labels.size(0)

    train_loss = running_loss / len(train_dl)
    val_acc    = correct / total_n if total_n else 0.0
    scheduler.step()
    history['loss'].append(train_loss); history['val_acc'].append(val_acc)
    print(f"Epoch {epoch:3d}/{CNN_EPOCHS}  loss={train_loss:.4f}  val_acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(patch_cnn.state_dict(), CNN_WEIGHTS)
        print(f"  -> Best saved -> {CNN_WEIGHTS}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['loss']); ax1.set_title('Training loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['val_acc']); ax2.set_title('Validation accuracy'); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()
print(f"Training complete. Best val accuracy: {best_val_acc:.4f}")


### 3C. EMA Smoother & BiLSTM (Temporal Smoothing)

Two smoothers are available:

| Smoother | Training needed? | Latency | Use |
|---|---|---|---|
| **EMASmoother** | No | < 1 ms | Recommended for all edge deployments |
| **BiLSTM** | Yes (labelled video sequences) | ~3 ms | Only if you have training sequence data |

The EMA smoother is used by default in both pipeline inference loops.


In [ ]:
class EMASmoother:
    '''
    Exponential Moving Average smoother for the ON-class probability.
    Tune alpha on your validation video:
      alpha=0.2  ->  resistant to flicker, slower to react
      alpha=0.5  ->  faster to react, more sensitive to noise
    '''
    def __init__(self, alpha: float = 0.35, threshold: float = 0.52, min_frames: int = 6):
        self.alpha = alpha; self.threshold = threshold; self.min_frames = min_frames
        self.ema = 0.5; self.n = 0

    def reset(self):
        self.ema = 0.5; self.n = 0

    def update(self, on_prob: float):
        self.ema = self.alpha * on_prob + (1.0 - self.alpha) * self.ema
        self.n  += 1
        if self.n < self.min_frames:
            return 'WARMING', self.ema
        return ('ON' if self.ema >= self.threshold else 'OFF'), self.ema

smoother = EMASmoother(alpha=0.35, threshold=0.52, min_frames=6)
print("EMA smoother ready (alpha=0.35, threshold=0.52, warm-up=6 frames)")


class FusionBiLSTM(nn.Module):
    '''
    Optional BiLSTM temporal smoother.
    Requires labelled sequence data — use EMASmoother for edge deployment.
    '''
    def __init__(self, input_dim=FUSED_DIM, hidden_dim=LSTM_HIDDEN,
                 num_layers=LSTM_LAYERS, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.proj   = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU())
        self.bilstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head   = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        x = self.proj(x); out, _ = self.bilstm(x)
        return self.head(out[:, -1, :])

bilstm_model = FusionBiLSTM().to(DEVICE)
print(f"BiLSTM — {count_parameters(bilstm_model):,} parameters (optional)")


EMA smoother ready (alpha=0.35, threshold=0.52, warm-up=6 frames)
BiLSTM — 174,850 parameters (optional)


In [ ]:
import uuid

# ── Sequence feature extraction — compatible with build_fusion_vector() ───────
# This cell must be run AFTER Section 8 (inference_utils) is defined,
# because it calls extract_torso_roi, run_yolo_on_roi, run_patch_cnn,
# geometric_prior_score, compute_motion, and build_fusion_vector.

SEQUENCE_DIR = f"{RUN_DIR}/sequences"   # uses global RUN_DIR from config cell
os.makedirs(SEQUENCE_DIR + "/train", exist_ok=True)
os.makedirs(SEQUENCE_DIR + "/valid", exist_ok=True)


def extract_sequence_features(
    video_path: str,
    label: int,           # 1 = ON, 0 = OFF
    save_dir: str,
    skip_frames: int = 1, # set >1 to subsample long videos
) -> None:
    """
    Process a labelled video end-to-end through the full inference pipeline
    and save the resulting per-frame fused feature vectors as a .npy file.

    The saved array has shape (T, FUSED_DIM=6) — identical to what
    build_fusion_vector() produces at inference time — so SequenceDataset
    and the BiLSTM DataLoader receive correctly-shaped tensors.

    Feature vector layout (matches build_fusion_vector):
        [0] yolo_pred   — binary YOLO decision (0/1)
        [1] yolo_conf   — YOLO ON-class confidence (0-1)
        [2] cnn_pred    — binary CNN decision (0/1)
        [3] cnn_conf    — CNN ON-class confidence (0-1)
        [4] motion      — normalised frame-difference magnitude (0-1)
        [5] prior_score — RANSAC geometric belt-line score (0-1)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[WARN] Cannot open video: {video_path}")
        return

    sequence   = []          # list of 6-D np.float32 vectors
    prev_frame = None        # kept in BGR for compute_motion
    frame_idx  = 0

    while True:
        ret, frame = cap.read()  # frame is BGR — do NOT convert to RGB
        if not ret:
            break

        frame_idx += 1
        if frame_idx % skip_frames != 0:
            continue

        # ── Stage 1: pose-guided ROI (expects BGR) ───────────────────────────
        roi, bbox, keypoints = extract_torso_roi(frame, padding=TORSO_PAD)

        # ── Stage 2: YOLO detection on ROI ───────────────────────────────────
        yolo_pred, yolo_conf = run_yolo_on_roi(yolo_model, roi)

        # ── Stage 3: patch CNN classification on ROI ─────────────────────────
        cnn_pred, cnn_conf = run_patch_cnn(patch_cnn, roi)

        # ── Stage 4: geometric prior (RANSAC) ────────────────────────────────
        prior_score = 0.5   # neutral default when prior is disabled
        if USE_GEO_PRIOR and roi is not None and roi.size > 0:
            prior_score = geometric_prior_score(
                roi      = roi,
                keypoints= keypoints,
                roi_bbox = bbox if bbox else (0, 0, frame.shape[1], frame.shape[0]),
            )

        # ── Stage 5: motion (requires previous BGR frame) ────────────────────
        motion = compute_motion(prev_frame, frame)  # returns 0.0 on first frame

        # ── Stage 6: build the 6-D fused vector (single source of truth) ─────
        fv = build_fusion_vector(
            yolo_pred, yolo_conf,
            cnn_pred,  cnn_conf,
            motion, prior_score,
        )  # returns list[float] of length FUSED_DIM=6

        sequence.append(np.array(fv, dtype=np.float32))
        prev_frame = frame.copy()

    cap.release()

    if len(sequence) == 0:
        print(f"[WARN] No frames extracted from {video_path}")
        return

    # Shape: (T, 6) — T varies per video; SequenceDataset will pad/truncate
    arr  = np.array(sequence, dtype=np.float32)   # (T, FUSED_DIM)
    name = f"seq_{'ON' if label == 1 else 'OFF'}_{uuid.uuid4().hex}.npy"
    np.save(os.path.join(save_dir, name), arr)
    print(f"Saved: {name}  shape={arr.shape}  label={'ON' if label==1 else 'OFF'}")


# ── Example usage ─────────────────────────────────────────────────────────────
# Run this cell AFTER training both YOLO and PatchCNN (sections 3 & 5).
# Point video_list at your labelled clips; split into train/ and valid/ manually.
#
#video_list = [
#    ("valid_on.mp4",    1),   # seatbelt ON
#    ("valid_off.mp4", 0),   # seatbelt OFF
#]
#for video_path, label in video_list:
#    extract_sequence_features(video_path, label, SEQUENCE_DIR + "/valid")


print("extract_sequence_features() defined ✓")
print(f"  Fused vector dim : {FUSED_DIM}  (matches build_fusion_vector)")
print(f"  Sequence length  : {SEQUENCE_LEN} frames (global config, used by SequenceDataset)")


extract_sequence_features() defined ✓
  Fused vector dim : 6  (matches build_fusion_vector)
  Sequence length  : 16 frames (global config, used by SequenceDataset)


In [ ]:
# ── SequenceDataset & BiLSTM training loop ───────────────────────────────────
# Expects .npy files produced by extract_sequence_features() (cell above).
# Each file: shape (T, FUSED_DIM=6), filename contains 'ON' or 'OFF'.

class SequenceDataset(Dataset):
    """
    Loads per-video fused-feature sequences from .npy files and pads/truncates
    them to a fixed length (SEQUENCE_LEN) so DataLoader can batch them.

    Expected file layout:
        seq_dir/
            seq_ON_<uuid>.npy    shape (T, 6)  → label 1
            seq_OFF_<uuid>.npy   shape (T, 6)  → label 0

    The second axis MUST equal FUSED_DIM (6).  Files written by the old
    extract_sequence_features() (dim=4) will raise a clear ValueError here
    rather than a cryptic collation crash inside DataLoader.
    """

    def __init__(self, seq_dir: str, seq_len: int = SEQUENCE_LEN):
        self.files   = sorted(Path(seq_dir).glob('*.npy'))
        self.seq_len = seq_len
        print(f"SequenceDataset: {len(self.files)} sequences from {seq_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        seq  = np.load(str(path)).astype(np.float32)   # expected: (T, FUSED_DIM)

        # ── Guard: ensure feature dimension matches global FUSED_DIM ─────────
        if seq.ndim != 2 or seq.shape[1] != FUSED_DIM:
            raise ValueError(
                f"Sequence file {path.name} has shape {seq.shape} but expected "
                f"(T, {FUSED_DIM}). Re-run extract_sequence_features() to "
                f"regenerate files with the correct 6-D fused vectors."
            )

        # ── Pad (front) or truncate (tail) to fixed SEQUENCE_LEN ─────────────
        T = len(seq)
        if T < self.seq_len:
            # Zero-pad at the start so the most recent frames are at the end
            pad = np.zeros((self.seq_len - T, FUSED_DIM), dtype=np.float32)
            seq = np.vstack([pad, seq])          # (seq_len, FUSED_DIM)
        else:
            seq = seq[-self.seq_len:]            # keep the LAST seq_len frames

        # ── Label from filename: seq_ON_*.npy → 1, seq_OFF_*.npy → 0 ─────────
        # Use stem without UUID so 'ON' in 'seq_ON_abc...' is unambiguous.
        label = 1 if '_ON_' in path.stem else 0
        return torch.from_numpy(seq), label      # (SEQUENCE_LEN, FUSED_DIM), int


def train_bilstm(seq_train_dir: str, seq_val_dir: str) -> float:
    """Train the BiLSTM smoother on pre-extracted fused feature sequences."""
    train_ds = SequenceDataset(seq_train_dir)
    val_ds   = SequenceDataset(seq_val_dir)

    # ── Guard: skip training if either split is empty ─────────────────────────
    if len(train_ds) == 0:
        print("[SKIP] No training sequences found. Run extract_sequence_features() first.")
        return 0.0
    if len(val_ds) == 0:
        print("[WARN] No validation sequences found — training without validation.")

    # batch_size capped at dataset size to avoid empty batches
    bs_train = min(32, len(train_ds))
    bs_val   = min(32, len(val_ds)) if len(val_ds) > 0 else 1

    train_dl = DataLoader(train_ds, batch_size=bs_train, shuffle=True,  drop_last=False)
    val_dl   = DataLoader(val_ds,   batch_size=bs_val,   shuffle=False, drop_last=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    scaler    = GradScaler()

    best_val_acc = 0.0

    for epoch in range(1, LSTM_EPOCHS + 1):
        # ── Training ──────────────────────────────────────────────────────────
        bilstm_model.train()
        total_loss = 0.0
        for seqs, labels in train_dl:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                logits = bilstm_model(seqs)       # (B, SEQUENCE_LEN, FUSED_DIM)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        # ── Validation ────────────────────────────────────────────────────────
        bilstm_model.eval()
        correct = total = 0
        if len(val_ds) > 0:
            with torch.no_grad():
                for seqs, labels in val_dl:
                    seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
                    preds = bilstm_model(seqs).argmax(1)
                    correct += (preds == labels).sum().item()
                    total   += labels.size(0)

        val_acc = correct / total if total > 0 else float('nan')
        scheduler.step()
        print(f"BiLSTM Epoch {epoch:3d}/{LSTM_EPOCHS}  "
              f"loss={total_loss / len(train_dl):.4f}  "
              f"val_acc={val_acc:.4f}")

        if not isinstance(val_acc, float) or val_acc > best_val_acc:
            best_val_acc = val_acc if isinstance(val_acc, float) else 0.0
            torch.save(bilstm_model.state_dict(), BILSTM_WEIGHTS)
            print(f"  ↳ New best BiLSTM → {BILSTM_WEIGHTS}")

    return best_val_acc


# ── Call train_bilstm() only when sequences exist ─────────────────────────────
# Run extract_sequence_features() in the cell above first, then uncomment:
#
#train_bilstm(
#    seq_train_dir = f"{RESULTS_PATH}/sequences/train",
#    seq_val_dir   = f"{RESULTS_PATH}/sequences/valid",
#)

print("SequenceDataset & train_bilstm() defined ✓")
print("→ Run extract_sequence_features() first, then uncomment train_bilstm().")


SequenceDataset & train_bilstm() defined ✓
→ Run extract_sequence_features() first, then uncomment train_bilstm().


## 4. Shared Utilities

**Run this entire section before Section 5 or Section 6.**  
All functions here are used by both pipelines.


### 4A. RANSAC Geometric Prior

Verifies that edge pixels in the torso ROI form a diagonal line consistent  
with a real seatbelt trajectory. Used as a tiebreaker when YOLO and CNN disagree.  
Only runs 30 iterations and is gated — skipped when both models already agree with high confidence.


In [ ]:
def extract_belt_candidates(roi: np.ndarray, t1: int = 50, t2: int = 150) -> np.ndarray:
    gray  = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, t1, t2)
    ys, xs = np.nonzero(edges)
    return np.column_stack([xs, ys]).astype(np.float32) if len(xs) else np.empty((0, 2), np.float32)

def ransac_line_fit(pts: np.ndarray, n_iter: int = 30,
                    dist_thresh: float = 4.0, min_inliers: int = 15):
    if len(pts) < min_inliers: return None, 0.0
    best_line, best_n = None, 0
    for _ in range(n_iter):
        idx    = np.random.choice(len(pts), 2, replace=False)
        p1, p2 = pts[idx[0]], pts[idx[1]]
        dx = p2[0] - p1[0]
        if abs(dx) < 1e-6: continue
        m  = (p2[1] - p1[1]) / dx; b = p1[1] - m * p1[0]
        dist = np.abs(m * pts[:, 0] - pts[:, 1] + b) / np.sqrt(m ** 2 + 1)
        n = (dist < dist_thresh).sum()
        if n > best_n: best_n = n; best_line = np.array([m, b])
    ratio = best_n / len(pts) if len(pts) else 0.0
    return (best_line if best_n >= min_inliers else None), ratio

def geometric_prior_score(roi: np.ndarray, keypoints: list,
                           roi_bbox: tuple, slope_tolerance: float = 0.5) -> float:
    '''
    Returns a [0,1] confidence score that the ROI contains a seatbelt-shaped diagonal line.
    Used as a tiebreaker when YOLO and CNN disagree.
    '''
    if roi is None or roi.size == 0: return 0.5
    cands = extract_belt_candidates(roi)
    if not len(cands): return 0.5
    line, inlier_ratio = ransac_line_fit(cands)
    if line is None: return 0.3
    expected_slope = -1.0
    if keypoints and len(keypoints) >= 4:
        x1r, y1r  = roi_bbox[0], roi_bbox[1]
        rs = (keypoints[1][0] - x1r, keypoints[1][1] - y1r)
        lh = (keypoints[2][0] - x1r, keypoints[2][1] - y1r)
        if abs(lh[0] - rs[0]) > 5:
            expected_slope = (lh[1] - rs[1]) / (lh[0] - rs[0])
    slope_err   = abs(line[0] - expected_slope)
    slope_score = max(0.0, 1.0 - slope_err / (slope_tolerance * 2))
    return float(np.clip(0.5 * inlier_ratio + 0.5 * slope_score, 0.0, 1.0))

print("Geometric prior (RANSAC, 30 iterations) ready")


Geometric prior (RANSAC, 30 iterations) ready


### 4B. Core Inference Helpers

Shared functions called by both pipeline inference loops.

In [ ]:
_val_tf = get_val_transforms(CNN_IMG_SIZE)

def load_inference_models(yolo_path=YOLO_WEIGHTS, cnn_path=CNN_WEIGHTS, lstm_path=BILSTM_WEIGHTS):
    '''Load YOLOv8, MobileNetV3 CNN, and BiLSTM from saved weights.'''
    yolo = YOLO(yolo_path); print(f"YOLO      <- {yolo_path}")
    cnn  = build_patch_cnn().to(DEVICE)
    cnn.load_state_dict(torch.load(cnn_path, map_location=DEVICE)); cnn.eval()
    print(f"PatchCNN  <- {cnn_path}")
    lstm = FusionBiLSTM().to(DEVICE)
    lstm.load_state_dict(torch.load(lstm_path, map_location=DEVICE)); lstm.eval()
    print(f"BiLSTM    <- {lstm_path}")
    return yolo, cnn, lstm

def run_yolo_on_roi(yolo_model, roi: np.ndarray, conf_thresh: float = CONF_THRESHOLD):
    '''
    Run YOLOv8 on a torso ROI crop.
    Returns (class_idx, confidence): 0=OFF, 1=ON.
    Defaults to (0, 0.0) when nothing passes the threshold.
    '''
    if roi is None or roi.size == 0: return 0, 0.0
    results = yolo_model(roi, verbose=False, conf=conf_thresh)
    best_conf, best_cls = 0.0, 0
    for r in results:
        if r.boxes is None: continue
        for box in r.boxes:
            cf = float(box.conf.item())
            if cf > best_conf: best_conf = cf; best_cls = int(box.cls.item())
    return best_cls, best_conf

def run_patch_cnn(cnn_model: nn.Module, roi: np.ndarray):
    '''
    Run MobileNetV3 on a BGR torso ROI.
    Returns (class_idx, on_class_confidence).
    '''
    if roi is None or roi.size == 0: return 0, 0.5
    rgb    = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    tensor = _val_tf(image=rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(cnn_model(tensor), dim=1)[0]
    return probs.argmax().item(), probs[1].item()

def compute_motion(prev: np.ndarray, curr: np.ndarray) -> float:
    if prev is None: return 0.0
    diff = cv2.absdiff(cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY),
                       cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY))
    return float(np.clip(diff.mean() / MOTION_THRESH, 0.0, 1.0))

def build_fusion_vector(yp, yc, cp, cc, motion, prior) -> list:
    return [float(yp), yc, float(cp), cc, motion, prior]

def predict_bilstm(model: nn.Module, buf: deque):
    seq = torch.from_numpy(np.array(list(buf), dtype=np.float32)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(seq), dim=1)[0]
    return probs.argmax().item(), probs[1].item()

print("Inference helpers ready")


Inference helpers ready


### 4C. Background Frame Capture Thread

Decouples video I/O from processing. The inference loop never stalls waiting  
for `cap.read()`. Stale frames are automatically dropped when processing falls behind.


In [ ]:
class FrameBuffer:
    '''
    Background thread that pre-fetches frames into a small queue.
    The inference loop calls buf.read() which blocks until a frame is ready.
    Stale frames are automatically dropped if processing falls behind.
    Usage: buf = FrameBuffer(source); frame = buf.read()
    '''
    def __init__(self, source, maxsize: int = 2):
        self.q   = _queue.Queue(maxsize=maxsize)
        self.cap = cv2.VideoCapture(source)
        threading.Thread(target=self._grab, daemon=True).start()

    def _grab(self):
        while True:
            ret, frame = self.cap.read()
            if not ret: self.q.put(None); break
            if self.q.full():
                try: self.q.get_nowait()
                except _queue.Empty: pass
            self.q.put(frame)

    def read(self): return self.q.get()

    @property
    def cap_props(self): return self.cap

print("FrameBuffer ready")


FrameBuffer ready


### 4D. ONNX Export & Runtime Helpers

Exporting the CNN to ONNX and running it via ONNX Runtime gives ~3 extra FPS  
on RPi4 without any accuracy loss.  
After training, run `export_patch_cnn_onnx()` and set `USE_ORT_CNN = True` in Pipeline B.


In [ ]:
import onnxruntime as ort

def export_patch_cnn_onnx(model: nn.Module, out_path: str, img_size: int = CNN_IMG_SIZE):
    '''Export MobileNetV3 to ONNX for ARM-optimised ORT inference.'''
    model.eval().cpu()
    torch.onnx.export(model, torch.randn(1, 3, img_size, img_size), out_path,
                      input_names=['input'], output_names=['logits'],
                      dynamic_axes={'input': {0: 'batch'}}, opset_version=17)
    print(f"CNN ONNX saved -> {out_path}")
    model.to(DEVICE)

def make_ort_session(onnx_path: str) -> ort.InferenceSession:
    '''ORT session tuned for RPi4: 4 threads, XNNPACK-fused CPU EP.'''
    opts = ort.SessionOptions()
    opts.intra_op_num_threads     = 4
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(onnx_path, sess_options=opts, providers=['CPUExecutionProvider'])

_ort_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_ort_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def run_patch_cnn_ort(session, roi: np.ndarray, img_size: int = CNN_IMG_SIZE):
    '''Drop-in for run_patch_cnn() — uses ONNX Runtime (~3 FPS faster on RPi4).'''
    if roi is None or roi.size == 0: return 0, 0.5
    rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    img = cv2.resize(rgb, (img_size, img_size)).astype(np.float32) / 255.0
    t   = ((img - _ort_mean) / _ort_std).transpose(2, 0, 1)[None]
    logits = session.run(None, {'input': t})[0]
    exp = np.exp(logits - logits.max()); probs = exp / exp.sum()
    return int(probs[0].argmax()), float(probs[0, 1])

# Uncomment after training to export, then set USE_ORT_CNN = True in Pipeline B:
# export_patch_cnn_onnx(patch_cnn, CNN_ONNX_PATH)
# cnn_ort_session = make_ort_session(CNN_ONNX_PATH)
print("ONNX export helpers ready")


ONNX export helpers ready


## 5. Pipeline A — YOLOv5 Full-Frame ROI + CNN

```
Frame
 |
 |--[1] YOLOv5 (full frame) -> seatbelt bounding boxes
 |       Single-class model: fires when a seatbelt strap is visible.
 |
 |--[2] Crop each YOLO box -> MobileNetV3 CNN -> ON / OFF probability
 |       Refines the raw YOLO detection with texture-level classification.
 |
 +--[3] Majority-vote smoothing over last 5 frames -> final label
```

### When to use Pipeline A
- Camera faces the driver directly (minimal angle)
- Seatbelt strap is consistently visible and unoccluded
- You want per-detection bounding boxes in the output video

### Known limitations
- If YOLOv5 misses the strap entirely the CNN is never called
- No pose context — a bag strap at the right angle can fool it
- Majority-vote can flip rapidly on borderline frames


### 5A. Probe your YOLOv5 model class mapping

Run this cell before using Pipeline A.  
A single-class YOLOv5 model outputs class 0 for every detection — confidence is the only signal.


In [ ]:
# ── Inspect the YOLOv5 model class mapping before using Pipeline A ────────────
from ultralytics import YOLO as _YOLO
_probe = _YOLO(YOLO_WEIGHTS)
print("YOLOv8 names:", _probe.names)
# Good:  {0: 'off', 1: 'on'}  or  {0: 'no-seatbelt', 1: 'seatbelt'}
# Single-class: {0: 'seatbelt'}  -> set YOLOV5_SINGLE_CLASS = True below

# Adjust these two constants based on the probe output above:
YOLOV5_SINGLE_CLASS  = True    # True if model has only one class (seatbelt present)
PATCH_CONF_THRESHOLD = 0.65    # YOLOv5 confidence above which strap is considered present


YOLOv8 names: {0: 'no-seatbelt', 1: 'seatbelt'}


### 5B. Pipeline A functions + annotation

In [ ]:
def annotate_pipeline_a(
    frame: np.ndarray,
    smoothed_results: list,
    fps: float = 0.0,
) -> np.ndarray:
    '''
    Pipeline A annotation.

    For each detection: coloured box + label banner above the box + YOLO conf below.
    When nothing is detected: neutral grey status banner.

    Parameters
    ----------
    frame            : BGR frame (modified in place)
    smoothed_results : list of [x1, y1, x2, y2, yolo_conf, patch_cls, patch_conf]
    fps              : current frames per second
    '''
    H, W = frame.shape[:2]

    if not smoothed_results:
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (W, 50), (80, 80, 80), -1)
        cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
        cv2.putText(frame, "No seatbelt detected",
                    (10, 34), cv2.FONT_HERSHEY_DUPLEX, 0.9, (200, 200, 200), 2)
    else:
        for r in smoothed_results:
            x1, y1, x2, y2 = int(r[0]), int(r[1]), int(r[2]), int(r[3])
            yolo_conf       = float(r[4])
            patch_cls       = int(r[5])
            patch_conf      = float(r[6])

            color = (0, 200, 0) if patch_cls == 1 else (0, 0, 220)
            label = "Seatbelt ON" if patch_cls == 1 else "Seatbelt OFF"

            # Detection bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Coloured label banner directly above the box
            banner_y1 = max(0, y1 - 36)
            ov = frame.copy()
            cv2.rectangle(ov, (x1, banner_y1), (x2, y1), color, -1)
            cv2.addWeighted(ov, 0.70, frame, 0.30, 0, frame)
            cv2.putText(frame, f"{label}  ({patch_conf:.0%})",
                        (x1 + 5, max(15, y1 - 10)),
                        cv2.FONT_HERSHEY_DUPLEX, 0.60, (255, 255, 255), 2)

            # YOLO raw confidence beneath the box
            cv2.putText(frame, f"YOLO: {yolo_conf:.0%}",
                        (x1 + 5, y2 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1)

    # Pipeline badge (bottom-left)
    cv2.putText(frame, "Pipeline A: YOLOv5 ROI + CNN",
                (8, H - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
    # FPS (bottom-right)
    if fps > 0:
        cv2.putText(frame, f"FPS: {fps:.1f}",
                    (W - 110, H - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (200, 200, 200), 1)
    return frame


In [ ]:
def run_yolov5_on_frame(yolov5_model, frame: np.ndarray):
    '''YOLOv5 full-frame detection. Returns (class, confidence).'''
    img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    boxes    = yolov5_model(img_rgb).xyxy[0].cpu()
    best_conf = 0.0
    for box in boxes:
        cf = float(box[4])
        if cf > best_conf: best_conf = cf
    return (1 if best_conf >= PATCH_CONF_THRESHOLD else 0), best_conf


def yolov5_to_patch_cnn(yolov5_model, cnn_model: nn.Module, frame: np.ndarray) -> list:
    '''
    Run YOLOv5 to detect seatbelt regions, crop each box, classify with CNN.

    Returns a list of [x1, y1, x2, y2, yolo_conf, patch_cls, patch_conf].
    '''
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    boxes   = yolov5_model(img_rgb).xyxy[0].cpu()
    results = []
    for j in boxes:
        x1, y1, x2, y2, conf, _ = j.numpy()
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        roi = frame[y1:y2, x1:x2]
        if roi.size == 0: continue
        patch_cls, patch_conf = run_patch_cnn(cnn_model, roi)
        if patch_cls == 1 and patch_conf < PATCH_CONF_THRESHOLD:
            patch_cls = 0   # apply confidence gate
        results.append([x1, y1, x2, y2, float(conf), patch_cls, patch_conf])
    return results


# Majority-vote smoothing queue (last 5 frames)
_smooth_q_a = deque(maxlen=5)

def process_frame_pipeline_a(frame: np.ndarray, yolov5_model, cnn_model: nn.Module,
                              fps: float = 0.0) -> np.ndarray:
    '''Full Pipeline A frame processing: detect -> classify -> smooth -> annotate.'''
    raw = yolov5_to_patch_cnn(yolov5_model, cnn_model, frame)
    _smooth_q_a.append([(r[5], r[6]) for r in raw])

    smoothed = []
    for i, r in enumerate(raw):
        votes = [f[i][0] for f in _smooth_q_a if i < len(f)]
        confs = [f[i][1] for f in _smooth_q_a if i < len(f)]
        sm_cls  = max(set(votes), key=votes.count) if votes else r[5]
        sm_conf = sum(confs) / len(confs) if confs else r[6]
        smoothed.append([r[0], r[1], r[2], r[3], r[4], sm_cls, sm_conf])

    return annotate_pipeline_a(frame.copy(), smoothed, fps)

print("Pipeline A functions ready")


Pipeline A functions ready


### 5C. Pipeline A inference loop

In [ ]:
import torch, datetime as dt

INPUT_VIDEO  = "valid_off.mp4"          # <- change to your test video
OUTPUT_DIR   = "output"; os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE  = os.path.join(OUTPUT_DIR, f"pipelineA_{dt.datetime.now().strftime('%H%M%S')}.mp4")

# ── Load models ───────────────────────────────────────────────────────────────
yolov8_model, patch_cnn, _ = load_inference_models()
patch_cnn.eval()

yolov5_model = torch.hub.load(
    "ultralytics/yolov5", "custom", path=YOLOV5_WEIGHTS, force_reload=False
)
yolov5_model.eval()
_smooth_q_a.clear()

# ── Video I/O ─────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened(): raise RuntimeError(f"Cannot open {INPUT_VIDEO}")
fw, fh  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_src = cap.get(cv2.CAP_PROP_FPS) or 20
writer  = cv2.VideoWriter(OUTPUT_FILE, cv2.VideoWriter_fourcc(*'mp4v'), fps_src, (fw, fh))

fps_timer = time.perf_counter(); fps = 0.0; frame_n = 0
print(f"Processing {INPUT_VIDEO}  ->  {OUTPUT_FILE}")

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_n += 1
    annotated = process_frame_pipeline_a(frame, yolov5_model, patch_cnn, fps)
    now = time.perf_counter(); fps = 1.0 / max(now - fps_timer, 1e-6); fps_timer = now
    if frame_n % 30 == 0: print(f"Frame {frame_n}  FPS={fps:.1f}")
    writer.write(annotated)

cap.release(); writer.release()
print(f"Pipeline A complete -> {OUTPUT_FILE}")


YOLO      <- runs/seatbelt_edge_v2/best.pt
PatchCNN  <- runs/seatbelt_edge_v2/patch_cnn.pt
BiLSTM    <- runs/seatbelt_edge_v2/bilstm.pt


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master


requirements: Ultralytics requirement ['urllib3>=2.6.0 ; python_version > "3.8"'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ requirements: ❌ Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
error: Failed to parse: `urllib3>=2.6.0 ; python_version > 3.8`
  Caused by: Expected a quoted string or a valid marker name, found `3.8`
urllib3>=2.6.0 ; python_version > 3.8
            

YOLOv5 🚀 2026-4-1 Python-3.12.13 torch-2.10.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 1760518 parameters, 0 gradients, 4.1 GFLOPs
Adding AutoShape... 


Processing valid_off.mp4  ->  output/pipelineA_000429.mp4
Frame 30  FPS=7.9
Frame 60  FPS=8.0
Frame 90  FPS=5.5
Frame 120  FPS=8.2
Frame 150  FPS=7.3
Frame 180  FPS=3.3
Frame 210  FPS=8.2
Frame 240  FPS=8.1
Frame 270  FPS=8.1
Frame 300  FPS=5.5
Frame 330  FPS=7.7
Frame 360  FPS=7.2
Frame 390  FPS=4.9
Frame 420  FPS=8.1
Frame 450  FPS=8.2
Frame 480  FPS=8.2
Frame 510  FPS=5.8
Frame 540  FPS=7.4
Frame 570  FPS=8.0
Frame 600  FPS=5.7
Frame 630  FPS=8.1
Frame 660  FPS=7.1
Frame 690  FPS=8.3
Frame 720  FPS=4.8
Frame 750  FPS=7.3
Frame 780  FPS=7.9
Frame 810  FPS=8.1
Frame 840  FPS=7.6
Frame 870  FPS=8.4
Frame 900  FPS=8.2
Frame 930  FPS=4.9
Frame 960  FPS=7.0
Frame 990  FPS=8.0
Frame 1020  FPS=8.1
Frame 1050  FPS=5.2
Frame 1080  FPS=8.2
Frame 1110  FPS=8.3
Frame 1140  FPS=5.6
Frame 1170  FPS=8.1
Frame 1200  FPS=7.9
Frame 1230  FPS=8.1
Frame 1260  FPS=5.3
Frame 1290  FPS=8.0
Frame 1320  FPS=7.9
Frame 1350  FPS=5.3
Frame 1380  FPS=7.3
Frame 1410  FPS=8.1
Frame 1440  FPS=8.2
Frame 1470  FPS=5.

## 6. Pipeline B — MediaPipe Pose + EMA Fusion

```
Frame
 |
 |--[1] MediaPipe Pose (lite) -> 4 torso keypoints -> cropped ROI
 |       Shoulder + hip landmarks define a tight body-centred crop.
 |       Falls back to full frame if no person is detected.
 |
 |--[2] YOLOv8n on ROI -> yolo_pred, yolo_conf
 |
 |--[3] MobileNetV3 CNN on ROI -> cnn_pred, cnn_conf
 |
 |--[4] RANSAC Geometric Prior (gated) -> prior_score
 |       Only runs when YOLO and CNN disagree or both have low confidence.
 |
 +--[5] EMA Temporal Fusion -> final label
         on_prob = 0.6 x cnn_conf + 0.4 x yolo_conf
         Smoothed over time. Warm-up = 6 frames.
```

### When to use Pipeline B
- Camera is mounted at an angle (high dashcam, side-facing)
- Occlusions are frequent (steering wheel, arm, passenger)
- Multiple occupants may be in frame
- Edge hardware where robustness matters more than raw speed

### Known limitations
- Requires a detectable person — empty seat falls back to full frame
- MediaPipe lite can produce poor keypoints at extreme angles


### 6A. Pipeline B annotation (fixed)

The annotation has been rewritten to match Pipeline A's visual style:

- **Top banner**: coloured status (green = ON, red = OFF, grey = WARMING) with label + EMA confidence
- **Body**: torso ROI bounding box and four pose keypoint dots (shoulders + hips)
- **Bottom panel**: four model score cards — each shows the model name, its prediction,  
  and a confidence progress bar
- **Badge**: pipeline identifier bottom-left, FPS counter bottom-right


In [ ]:
def annotate_pipeline_b(
    frame       : np.ndarray,
    label       : str,
    conf        : float,
    bbox        : tuple | None,
    kps         : list,
    fps         : float = 0.0,
    yolo_pred   : int   = -1,
    yolo_conf   : float = 0.0,
    cnn_pred    : int   = -1,
    cnn_conf    : float = 0.0,
    prior_score : float = 0.5,
) -> np.ndarray:
    '''
    Pipeline B annotation.

    Layout
    ------
    Top    : coloured status banner showing the final EMA label + confidence.
    Body   : torso ROI bounding box  +  four pose keypoint dots (shoulders/hips).
    Bottom : four model score cards  [YOLO | CNN | GEO | EMA]
             each card: model name, predicted label, confidence progress bar.
    Corner : pipeline badge (bottom-left of cards), FPS counter (bottom-right).

    Parameters
    ----------
    frame       : BGR frame (modified in place)
    label       : "ON", "OFF", or "WARMING"
    conf        : EMA confidence (0-1)
    bbox        : (x1, y1, x2, y2) torso ROI in pixel coordinates
    kps         : [(x,y)] pose keypoints — [L-shoulder, R-shoulder, L-hip, R-hip]
    fps         : current frames per second
    yolo_pred   : YOLOv8 prediction index (0=OFF, 1=ON, -1=not run)
    yolo_conf   : YOLOv8 ON-class confidence
    cnn_pred    : CNN prediction index
    cnn_conf    : CNN ON-class confidence
    prior_score : RANSAC geometric prior score (0-1)
    '''
    _COLORS = {'ON': (0, 200, 0), 'OFF': (0, 0, 220), 'WARMING': (160, 160, 160)}
    color   = _COLORS.get(label, (160, 160, 160))
    H, W    = frame.shape[:2]

    # ── Torso ROI bounding box ────────────────────────────────────────────────
    if bbox:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # ── Pose keypoints (shoulder + hip landmarks) ─────────────────────────────
    for kp in kps:
        cv2.circle(frame, kp, 5, (255, 200, 0), -1)

    # ── Top status banner — final EMA result ──────────────────────────────────
    ov = frame.copy()
    cv2.rectangle(ov, (0, 0), (W, 50), color, -1)
    cv2.addWeighted(ov, 0.60, frame, 0.40, 0, frame)
    cv2.putText(frame, f"Seatbelt {label}  ({conf:.0%})",
                (10, 34), cv2.FONT_HERSHEY_DUPLEX, 0.9, (255, 255, 255), 2)

    # ── Per-model score cards ─────────────────────────────────────────────────
    models = [
        ("YOLO", CLASS_NAMES[yolo_pred] if yolo_pred >= 0 else "N/A", yolo_conf),
        ("CNN",  CLASS_NAMES[cnn_pred]  if cnn_pred  >= 0 else "N/A", cnn_conf),
        ("GEO",  "ON" if prior_score >= 0.5 else "OFF",               prior_score),
        ("EMA",  label if label != "WARMING" else "--",               conf),
    ]
    cell_w, cell_h = 100, 56
    px, py = 8, H - cell_h - 10

    for i, (name, pred, score) in enumerate(models):
        cx, cy     = px + i * (cell_w + 6), py
        cell_color = (34, 139, 34) if pred == "ON" else ((0, 0, 178) if pred == "OFF" else (70, 70, 70))

        ov = frame.copy()
        cv2.rectangle(ov,    (cx, cy), (cx + cell_w, cy + cell_h), cell_color, -1)
        cv2.addWeighted(ov, 0.55, frame, 0.45, 0, frame)
        cv2.rectangle(frame, (cx, cy), (cx + cell_w, cy + cell_h), (210, 210, 210), 1)

        cv2.putText(frame, name,
                    (cx + 6, cy + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (210, 210, 210), 1)
        cv2.putText(frame, pred,
                    (cx + 6, cy + 36), cv2.FONT_HERSHEY_DUPLEX,  0.60, (255, 255, 255), 2)

        # Confidence progress bar (bottom of card)
        bmax = cell_w - 8
        blen = int(bmax * min(max(score, 0.0), 1.0))
        by   = cy + cell_h - 7
        cv2.rectangle(frame, (cx + 4, by), (cx + 4 + bmax, by + 4), (50, 50, 50),      -1)
        cv2.rectangle(frame, (cx + 4, by), (cx + 4 + blen, by + 4), (255, 255, 255),   -1)

    # ── Pipeline badge + FPS ─────────────────────────────────────────────────
    badge_x = px + 4 * (cell_w + 6) + 6
    cv2.putText(frame, "Pipeline B: MediaPipe + EMA",
                (badge_x, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1)
    if fps > 0:
        cv2.putText(frame, f"FPS: {fps:.1f}",
                    (W - 110, H - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (200, 200, 200), 1)
    return frame

print("annotate_pipeline_b ready")


annotate_pipeline_b ready


### 6B. Pipeline B inference loop

In [ ]:
import datetime as dt

INPUT_VIDEO  = "valid_on.mp4"            # <- change to your test video
OUTPUT_DIR   = "output"; os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, f"pipelineB_{dt.datetime.now().strftime('%H%M%S')}.mp4")
RESIZE_W     = 640
USE_ORT_CNN  = False   # set True after export_patch_cnn_onnx() + make_ort_session()

# ── Load models ───────────────────────────────────────────────────────────────
yolo_model, patch_cnn, _ = load_inference_models()
patch_cnn.eval()
smoother.reset()

# ── Open video via FrameBuffer (background capture thread) ────────────────────
# -----------------------------
# Video setup (FIXED)
# -----------------------------
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open {INPUT_VIDEO}")

fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_src = cap.get(cv2.CAP_PROP_FPS)

# Fix FPS if broken
if fps_src is None or fps_src <= 1:
    fps_src = 20

# Resize safely
if RESIZE_W is not None:
    scale = RESIZE_W / fw
    out_w = RESIZE_W
    out_h = int(fh * scale)
else:
    out_w, out_h = fw, fh

# Create writer with EXACT SAME SIZE as frames
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_src,
    (out_w, out_h)
)
label = 'WARMING'; final_conf = 0.0; bbox = None; keypoints = []
print(f"Processing {INPUT_VIDEO}  ->  {OUTPUT_VIDEO}")


while True:
    ret, frame = cap.read()
    if not ret: break
    frame_n += 1

    frame = cv2.resize(frame, (out_w, out_h))

    # Skip-frames: write last annotation, skip processing ─────────────────────
    if SKIP_FRAMES > 1 and frame_n % SKIP_FRAMES != 0:
        annotated = annotate_pipeline_b(frame, label, final_conf, bbox, keypoints, fps)
        writer.write(annotated); continue

    # Stage 1: MediaPipe torso ROI ────────────────────────────────────────────
    roi, bbox, keypoints = extract_torso_roi(frame, padding=TORSO_PAD)
    if roi is None or roi.size == 0:
        roi = frame.copy(); bbox = (0, 0, out_w, out_h); keypoints = []

    # Stage 2: YOLO on ROI ────────────────────────────────────────────────────
    yolo_pred, yolo_conf = run_yolo_on_roi(yolo_model, roi)

    # Stage 3: CNN on ROI ─────────────────────────────────────────────────────
    if USE_ORT_CNN:
        cnn_pred, cnn_conf = run_patch_cnn_ort(cnn_ort_session, roi)
    else:
        cnn_pred, cnn_conf = run_patch_cnn(patch_cnn, roi)

    # Stage 4: Geometric prior (only runs when YOLO and CNN disagree) ─────────
    prior_score = 0.5
    if USE_GEO_PRIOR:
        both_agree = (yolo_pred == cnn_pred) and (max(yolo_conf, cnn_conf) > 0.75)
        if both_agree:
            prior_score = 1.0 if yolo_pred == 1 else 0.0
        else:
            prior_score = geometric_prior_score(roi, keypoints, bbox)

    # Stage 5: EMA temporal fusion ────────────────────────────────────────────
    on_prob           = 0.6 * cnn_conf + 0.4 * yolo_conf
    label, final_conf = smoother.update(on_prob)

    # Stage 6: Annotate ───────────────────────────────────────────────────────
    now = time.perf_counter(); fps = 1.0 / max(now - fps_timer, 1e-6); fps_timer = now

    annotated = annotate_pipeline_b(
        frame.copy(), label, final_conf, bbox, keypoints, fps,
        yolo_pred, yolo_conf, cnn_pred, cnn_conf, prior_score,
    )
    writer.write(annotated)
    if frame_n % 30 == 0: print(f"Frame {frame_n}  FPS={fps:.1f}  label={label}")

writer.release()
print(f"Pipeline B complete -> {OUTPUT_VIDEO}")


YOLO      <- runs/seatbelt_edge_v2/best.pt
PatchCNN  <- runs/seatbelt_edge_v2/patch_cnn.pt
BiLSTM    <- runs/seatbelt_edge_v2/bilstm.pt
Processing valid_on.mp4  ->  output/pipelineB_233431.mp4
Frame 1770  FPS=0.0  label=WARMING
Frame 1800  FPS=3.5  label=ON
Frame 1830  FPS=3.4  label=ON
Frame 1860  FPS=2.7  label=ON
Frame 1890  FPS=3.3  label=OFF
Frame 1920  FPS=5.3  label=OFF
Frame 1950  FPS=2.2  label=ON
Frame 1980  FPS=3.0  label=ON
Frame 2010  FPS=3.3  label=ON
Frame 2040  FPS=2.6  label=ON
Frame 2070  FPS=3.2  label=ON
Frame 2100  FPS=2.9  label=ON
Frame 2130  FPS=2.3  label=ON
Frame 2160  FPS=3.4  label=OFF
Frame 2190  FPS=3.0  label=ON
Frame 2220  FPS=2.1  label=OFF
Frame 2250  FPS=3.2  label=OFF
Frame 2280  FPS=3.5  label=OFF
Frame 2310  FPS=2.6  label=ON
Frame 2340  FPS=4.1  label=ON
Frame 2370  FPS=3.2  label=OFF
Frame 2400  FPS=2.5  label=OFF
Frame 2430  FPS=2.9  label=OFF
Frame 2460  FPS=3.2  label=OFF
Frame 2490  FPS=2.3  label=ON
Frame 2520  FPS=3.8  label=ON
Frame 2550  

In [ ]:
import datetime as dt
#bilstm
INPUT_VIDEO  = "valid_off.mp4"            # <- change to your test video
OUTPUT_DIR   = "output"; os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, f"pipelineC_{dt.datetime.now().strftime('%H%M%S')}.mp4")
RESIZE_W     = 640
USE_ORT_CNN  = False   # set True after export_patch_cnn_onnx() + make_ort_session()

# ── Load models ───────────────────────────────────────────────────────────────
yolo_model, patch_cnn, _ = load_inference_models()
patch_cnn.eval()
smoother.reset()
seq_buffer = deque(maxlen=SEQUENCE_LEN)   # rolling fused-feature window
prev_frame = None
# ── Open video via FrameBuffer (background capture thread) ────────────────────
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open {INPUT_VIDEO}")

fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_src = cap.get(cv2.CAP_PROP_FPS)

# Fix FPS if broken
if fps_src is None or fps_src <= 1:
    fps_src = 20

# Resize safely
if RESIZE_W is not None:
    scale = RESIZE_W / fw
    out_w = RESIZE_W
    out_h = int(fh * scale)
else:
    out_w, out_h = fw, fh

# Create writer with EXACT SAME SIZE as frames
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_src,
    (out_w, out_h)
)
label = 'WARMING'; final_conf = 0.0; bbox = None; keypoints = []
print(f"Processing {INPUT_VIDEO}  ->  {OUTPUT_VIDEO}")


while True:
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    frame = cv2.resize(frame, (out_w, out_h))

    # Skip-frames: write last annotation, skip processing ─────────────────────
    if SKIP_FRAMES > 1 and frame_idx % SKIP_FRAMES != 0:
        annotated = annotate_pipeline_b(frame, label, final_conf, bbox, keypoints, fps)
        writer.write(annotated); continue

    # Stage 1: MediaPipe torso ROI ────────────────────────────────────────────
    roi, bbox, keypoints = extract_torso_roi(frame, padding=TORSO_PAD)
    if roi is None or roi.size == 0:
        roi = frame.copy(); bbox = (0, 0, out_w, out_h); keypoints = []

    # Stage 2: YOLO on ROI ────────────────────────────────────────────────────
    yolo_pred, yolo_conf = run_yolo_on_roi(yolo_model, roi)

    # Stage 3: CNN on ROI ─────────────────────────────────────────────────────
    if USE_ORT_CNN:
        cnn_pred, cnn_conf = run_patch_cnn_ort(cnn_ort_session, roi)
    else:
        cnn_pred, cnn_conf = run_patch_cnn(patch_cnn, roi)

    # Stage 4: Geometric prior (only runs when YOLO and CNN disagree) ─────────
    prior_score = 0.5
    if USE_GEO_PRIOR:
        both_agree = (yolo_pred == cnn_pred) and (max(yolo_conf, cnn_conf) > 0.75)
        if both_agree:
            prior_score = 1.0 if yolo_pred == 1 else 0.0
        else:
            prior_score = geometric_prior_score(roi, keypoints, bbox)

    # ════════════════════════════════════════════════════════════════════════════
    # STAGE 5: Build fused feature vector
    # ════════════════════════════════════════════════════════════════════════════
    motion = compute_motion(prev_frame, frame)
    fv     = build_fusion_vector(
        yolo_pred, yolo_conf,
        cnn_pred,  cnn_conf,
        motion, prior_score,
    )
    seq_buffer.append(fv)
    # ════════════════════════════════════════════════════════════════════════════
    # STAGE 6: BiLSTM temporal prediction
    # ════════════════════════════════════════════════════════════════════════════
    if len(seq_buffer) == SEQUENCE_LEN:
        final_pred, final_conf = predict_bilstm(bilstm_model, seq_buffer)
        label = CLASS_NAMES[final_pred]   # 'ON' or 'OFF'
    else:
        label      = 'WARMING'
        final_conf = 0.0

    # Stage 6: Annotate ───────────────────────────────────────────────────────
    now = time.perf_counter(); fps = 1.0 / max(now - fps_timer, 1e-6); fps_timer = now

    annotated = annotate_pipeline_b(
        frame.copy(), label, final_conf, bbox, keypoints, fps,
        yolo_pred, yolo_conf, cnn_pred, cnn_conf, prior_score,
    )
    writer.write(annotated)
    if frame_idx % 30 == 0: print(f"Frame {frame_idx}  FPS={fps:.1f}  label={label}")
    prev_frame = frame.copy()

writer.release()
print(f"Pipeline B complete -> {OUTPUT_VIDEO}")


YOLO      <- runs/seatbelt_edge_v2/best.pt
PatchCNN  <- runs/seatbelt_edge_v2/patch_cnn.pt
BiLSTM    <- runs/seatbelt_edge_v2/bilstm.pt
Processing valid_off.mp4  ->  output/pipelineC_235043.mp4
Frame 2730  FPS=0.0  label=WARMING
Frame 2760  FPS=2.7  label=ON
Frame 2790  FPS=2.1  label=ON
Frame 2820  FPS=3.1  label=ON
Frame 2850  FPS=3.3  label=ON
Frame 2880  FPS=2.2  label=ON
Frame 2910  FPS=3.2  label=ON
Frame 2940  FPS=3.5  label=ON
Frame 2970  FPS=2.0  label=ON
Frame 3000  FPS=4.5  label=ON
Frame 3030  FPS=3.9  label=ON
Frame 3060  FPS=2.2  label=ON
Frame 3090  FPS=3.1  label=ON
Frame 3120  FPS=3.2  label=ON
Frame 3150  FPS=2.3  label=ON
Frame 3180  FPS=3.3  label=ON
Frame 3210  FPS=3.4  label=ON
Frame 3240  FPS=2.3  label=ON
Frame 3270  FPS=3.4  label=ON
Frame 3300  FPS=3.1  label=ON
Frame 3330  FPS=2.2  label=ON
Frame 3360  FPS=3.7  label=ON
Frame 3390  FPS=3.7  label=ON
Frame 3420  FPS=2.4  label=ON
Frame 3450  FPS=3.1  label=ON
Frame 3480  FPS=3.3  label=ON
Frame 3510  FPS=2.1  

## 7. Pipeline Comparison

### Architecture

| | Pipeline A | Pipeline B |
|---|---|---|
| **ROI method** | YOLOv5 full-frame detection | MediaPipe pose (torso keypoints) |
| **ROI quality** | Tight — seatbelt region only | Generous — full torso + padding |
| **Classifier** | MobileNetV3 CNN | MobileNetV3 CNN (same weights) |
| **Second signal** | — | YOLOv8n on ROI |
| **Geometric check** | — | RANSAC belt-line (gated) |
| **Smoother** | Majority vote (5 frames) | EMA (alpha=0.35, warm-up 6 frames) |

### Performance on RPi 4

| Stage | Pipeline A | Pipeline B |
|---|---|---|
| ROI extraction | ~35 ms (YOLOv5 full frame) | ~10 ms (MediaPipe lite) |
| Classification | ~5 ms (CNN only) | ~5 ms (CNN) + ~30 ms (YOLO on ROI) |
| Smoother | < 1 ms | < 1 ms |
| **Total** | **~40 ms -> ~25 FPS** | **~45-50 ms -> ~20 FPS** |

### Accuracy trade-offs

| Scenario | Pipeline A | Pipeline B |
|---|---|---|
| Clear frontal view, visible strap | Excellent | Excellent |
| High/angled camera | May miss strap | Handles angle via pose |
| Partial occlusion (steering wheel) | Low confidence | CNN still classifies torso |
| Empty seat | No detection -> no label | Falls back to full frame |
| Multiple occupants | Separate box per person | Tracks one person at a time |

### Recommendation

Use **Pipeline A** when the camera directly faces the driver and the strap is consistently visible.  
Use **Pipeline B** when the camera is at an angle, occlusions are frequent, or you need  
greater robustness at the cost of ~5 ms extra latency per frame.


## 8. Evaluation & Metrics

Three evaluation levels:
1. **YOLO detection** — mAP50, precision, recall
2. **CNN classification** — accuracy, confusion matrix, ROC-AUC
3. **Pipeline B latency** — per-stage timing on a dummy frame


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

def evaluate_yolo(weights_path: str, data_yaml: str, conf: float = CONF_THRESHOLD):
    model   = YOLO(weights_path)
    metrics = model.val(data=data_yaml, conf=conf, verbose=False)
    print("YOLO Detection Metrics")
    print(f"  mAP@50    : {metrics.box.map50:.4f}")
    print(f"  mAP@50:95 : {metrics.box.map:.4f}")
    print(f"  Precision : {metrics.box.p.mean():.4f}")
    print(f"  Recall    : {metrics.box.r.mean():.4f}")

def evaluate_patch_cnn(cnn_model: nn.Module, val_loader: DataLoader):
    cnn_model.eval(); all_l, all_p, all_pr = [], [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            logits = cnn_model(imgs.to(DEVICE))
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            all_l.extend(labels.numpy()); all_p.extend(logits.argmax(1).cpu().numpy())
            all_pr.extend(probs[:, 1])
    all_l = np.array(all_l); all_p = np.array(all_p); all_pr = np.array(all_pr)
    print(classification_report(all_l, all_p, target_names=CLASS_NAMES, digits=4))
    auc = roc_auc_score(all_l, all_pr); print(f"ROC-AUC: {auc:.4f}")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    cm = confusion_matrix(all_l, all_p)
    ax1.imshow(cm, cmap='Blues'); ax1.set_title('Confusion matrix')
    for i in range(2):
        for j in range(2):
            ax1.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=16,
                     color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax1.set_xticks([0,1]); ax1.set_xticklabels(CLASS_NAMES)
    ax1.set_yticks([0,1]); ax1.set_yticklabels(CLASS_NAMES)
    fpr, tpr, _ = roc_curve(all_l, all_pr)
    ax2.plot(fpr, tpr, lw=2, label=f'AUC={auc:.3f}'); ax2.plot([0,1],[0,1],'k--',lw=1)
    ax2.set_title('ROC curve'); plt.tight_layout(); plt.show()
    return auc

def benchmark_pipeline_b(n_frames: int = 50):
    dummy  = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    timings = {'pose':[], 'yolo':[], 'cnn':[], 'prior':[], 'ema':[]}
    for _ in range(n_frames):
        t0 = time.perf_counter(); roi, bbox, kps = extract_torso_roi(dummy)
        timings['pose'].append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter(); yp, yc = run_yolo_on_roi(yolo_model, roi)
        timings['yolo'].append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter(); cp, cc = run_patch_cnn(patch_cnn, roi)
        timings['cnn'].append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter(); geometric_prior_score(roi, kps, bbox or (0,0,640,480))
        timings['prior'].append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter(); smoother.update(0.6*cc + 0.4*yc)
        timings['ema'].append((time.perf_counter()-t0)*1000)
    print("Pipeline B per-stage latency (ms)")
    total = 0.0
    for stage, vals in timings.items():
        m = np.mean(vals); total += m
        print(f"  {stage:<6}: {m:6.2f} ms  (+/-{np.std(vals):.2f})")
    print(f"  {'TOTAL':<6}: {total:6.2f} ms  ->  ~{1000/total:.1f} FPS")

evaluate_yolo(YOLO_WEIGHTS, DATASET_YAML)
evaluate_patch_cnn(patch_cnn, val_dl)
benchmark_pipeline_b()


Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.5 ms, read: 0.1±0.0 MB/s, size: 21.7 KB)
val: Scanning /content/drive/MyDrive/seatbelt-detection.v4i.yolov8/valid/labels.cache... 696 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 696/696 100.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 68% ━━━━━━━━──── 30/44 7.3s/it 3:03<1:42


KeyboardInterrupt: 

## 9. Export & Deployment

### Export formats

| Format | Target | Speed gain |
|---|---|---|
| ONNX FP32 @ 320 px | RPi 4 CPU via ORT | ~15 FPS over PyTorch |
| TorchScript FP16 | Jetson Nano GPU | ~2x PyTorch speed |

After exporting set `USE_ORT_CNN = True` in the Pipeline B loop.


In [ ]:
def export_yolo(weights_path: str):
    model = YOLO(weights_path)
    model.export(format='torchscript', half=True, imgsz=YOLO_IMGSZ, simplify=True)
    print("YOLO -> TorchScript FP16")
    model.export(format='onnx', half=False, imgsz=320, simplify=True, opset=17)
    print("YOLO -> ONNX FP32 @ 320px")

# Uncomment to export after training:
# export_yolo(YOLO_WEIGHTS)
# export_patch_cnn_onnx(patch_cnn, CNN_ONNX_PATH)
# Then set USE_ORT_CNN = True in the Pipeline B loop.
print("Export helpers ready (uncomment to run)")


Export helpers ready (uncomment to run)


## Summary & Deployment Checklist

### Training
- [ ] Section 1 — install packages and set your dataset path in config
- [ ] Section 3A — train YOLOv8n
- [ ] Section 3B — generate torso patches (both train + valid), then train CNN
- [ ] Section 3C — EMA smoother is always ready; skip BiLSTM unless you have sequence data

### Inference
- [ ] Run Section 4 in full before either pipeline
- [ ] Section 5 — run Pipeline A and check the output video
- [ ] Section 6 — run Pipeline B and check the output video
- [ ] Section 7 — compare and choose the pipeline for your camera setup
- [ ] Section 9 — export to ONNX, set `USE_ORT_CNN = True`

### RPi 4 tips
- `SKIP_FRAMES = 2` halves compute while carrying the last prediction
- `pose_landmarker_lite.task` is already set — never use the `_full` variant
- `YOLO_IMGSZ = 320` is already set — never use 640 on edge hardware
- Set `intra_op_num_threads = 4` in ORT (already in `make_ort_session`)
- Set `DISPLAY = False` — no monitor on RPi
